In [ ]:
import pandas as pd
import numpy as np
from src.preprocessing import knn_fill_na,create_lags,create_past_averages
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


pollen = pd.read_csv("data/pollen_berlin_all.csv")
ps_birch = pd.read_csv("data/ps_berlin_birch_final.csv")
ps_poac = pd.read_csv("data/ps_berlin_poac_final.csv")
symptoms = pd.read_csv("data/berlin_symptoms.csv")
ps_all = pd.concat([ps_birch, ps_poac], axis=0)
df = pd.concat([pollen,symptoms], axis=1)
df = knn_fill_na(df, "POAC",5)
df = knn_fill_na(df, "birch",5)

lags = [1,2,3,5,7]
df = create_lags(df, "averageOverallScoreWithMedication", lags)
df = create_lags(df, "birch", lags)
df = create_lags(df, "POAC", lags)
df = create_past_averages(df,"averageOverallScoreWithMedication",[3,5,7])
df = create_past_averages(df,"birch",[3,5,7])
df = create_past_averages(df,"POAC",[3,5,7])

lags = [1,2,3,5,7]
windows = [3,5,7]

max_lag = max(lags + windows)

df = df.iloc[max_lag:].reset_index(drop=True)


y = df["averageOverallScoreWithMedication"]

forbidden = [
    "averageOverallScoreWithMedication",
    "standardDeviationWithMedication",
    "averageOverallScoreWithoutMedication",
    "standardDeviationWithoutMedication"
]

X = df.drop(columns=forbidden)

# remove duplicated columns if they exist
df = df.loc[:, ~df.columns.duplicated()].copy()

# make sure date is datetime and sorted
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# target
target_col = "averageOverallScoreWithMedication"
y = df[target_col].copy()

# current-day columns we do NOT want as predictors
forbidden_current = [
    "averageOverallScoreWithMedication",
    "standardDeviationWithMedication",
    "averageOverallScoreWithoutMedication",
    "standardDeviationWithoutMedication"
]

# build X by dropping forbidden current-day columns
X = df.drop(columns=forbidden_current, errors="ignore").copy()

# remove date from predictors
X = X.drop(columns=["date"], errors="ignore")

# keep only numeric columns
X = X.select_dtypes(include=[np.number]).copy()

# optional: drop rows with any remaining NA
valid_idx = X.dropna().index.intersection(y.dropna().index)
X = X.loc[valid_idx].reset_index(drop=True)
y = y.loc[valid_idx].reset_index(drop=True)

# chronological train/test split
test_size = int(len(X) * 0.2)
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

# time-series CV only on training set
tscv = TimeSeriesSplit(n_splits=5)

# lasso pipeline for automatic lag selection
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=tscv, random_state=42, max_iter=20000))
])

# fit on training data
model.fit(X_train, y_train)

# predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# metrics
print("TRAIN METRICS")
print("R2  :", round(r2_score(y_train, y_pred_train), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 4))
print("MAE :", round(mean_absolute_error(y_train, y_pred_train), 4))

print("\nTEST METRICS")
print("R2  :", round(r2_score(y_test, y_pred_test), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 4))
print("MAE :", round(mean_absolute_error(y_test, y_pred_test), 4))

# selected features
lasso = model.named_steps["lasso"]
coef_series = pd.Series(lasso.coef_, index=X_train.columns)

selected_features = (
    coef_series[coef_series != 0]
    .sort_values(key=lambda s: np.abs(s), ascending=False)
)

print("\nSELECTED FEATURES")
print(selected_features)

lag_features = selected_features[selected_features.index.str.contains("lag|avg_prev", case=False, regex=True)]

print("\nSELECTED LAG / ROLLING FEATURES")
print(lag_features)

selected_cols = selected_features.index.tolist()
X_selected = X[selected_cols].copy()

print("\nFINAL SELECTED COLUMNS:")
for item in selected_cols:
    print(item)

In [ ]:


# remove duplicated columns if they exist
df = df.loc[:, ~df.columns.duplicated()].copy()

# make sure date is datetime and sorted
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# target
target_col = "averageOverallScoreWithMedication"
y = df[target_col].copy()

# current-day columns we do NOT want as predictors
forbidden_current = [
    "averageOverallScoreWithMedication",
    "standardDeviationWithMedication",
    "averageOverallScoreWithoutMedication",
    "standardDeviationWithoutMedication"
]

# build X by dropping forbidden current-day columns
X = df.drop(columns=forbidden_current, errors="ignore").copy()

# remove date from predictors
X = X.drop(columns=["date"], errors="ignore")

# keep only numeric columns
X = X.select_dtypes(include=[np.number]).copy()

# optional: drop rows with any remaining NA
valid_idx = X.dropna().index.intersection(y.dropna().index)
X = X.loc[valid_idx].reset_index(drop=True)
y = y.loc[valid_idx].reset_index(drop=True)

# chronological train/test split
test_size = int(len(X) * 0.2)
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]

# time-series CV only on training set
tscv = TimeSeriesSplit(n_splits=5)

# lasso pipeline for automatic lag selection
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=tscv, random_state=42, max_iter=20000))
])

# fit on training data
model.fit(X_train, y_train)

# predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# metrics
print("TRAIN METRICS")
print("R2  :", round(r2_score(y_train, y_pred_train), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 4))
print("MAE :", round(mean_absolute_error(y_train, y_pred_train), 4))

print("\nTEST METRICS")
print("R2  :", round(r2_score(y_test, y_pred_test), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 4))
print("MAE :", round(mean_absolute_error(y_test, y_pred_test), 4))

# selected features
lasso = model.named_steps["lasso"]
coef_series = pd.Series(lasso.coef_, index=X_train.columns)

selected_features = (
    coef_series[coef_series != 0]
    .sort_values(key=lambda s: np.abs(s), ascending=False)
)

print("\nSELECTED FEATURES")
print(selected_features)

lag_features = selected_features[selected_features.index.str.contains("lag|avg_prev", case=False, regex=True)]

print("\nSELECTED LAG / ROLLING FEATURES")
print(lag_features)

selected_cols = selected_features.index.tolist()
X_selected = X[selected_cols].copy()

print("\nFINAL SELECTED COLUMNS:")
for item in selected_cols:
    print(item)

TRAIN METRICS
R2  : 0.8735
RMSE: 0.4199
MAE : 0.3217

TEST METRICS
R2  : 0.6987
RMSE: 0.7661
MAE : 0.5539

SELECTED FEATURES
averageOverallScoreWithMedication_lag_1         0.591737
averageOverallScoreWithMedication_avg_prev_5    0.202360
averageOverallScoreWithMedication_avg_prev_3    0.131301
POAC                                            0.096518
averageOverallScoreWithMedication_lag_7         0.087390
averageOverallScoreWithMedication_lag_2         0.069415
birch                                           0.060467
POAC_avg_prev_5                                -0.044314
POAC_lag_3                                     -0.019897
averageOverallScoreWithMedication_lag_5         0.015814
POAC_lag_5                                     -0.013752
birch_lag_7                                    -0.012636
samples                                         0.012524
birch_avg_prev_7                               -0.006035
birch_lag_2                                    -0.005840
averageOverallScoreW